# 資料處理

## 檢查版本

In [11]:
import sys, xarray as xr
print("Python exe:", sys.executable)
print("Engines:", xr.backends.list_engines())

Python exe: /home/jundian/miniconda3/envs/geospatial-neural-adapter/bin/python
Engines: {'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'h5netcdf': <H5netcdfBackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


## 讀nc4

In [12]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:15<00:00, 36.41it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

In [13]:
# ===== 只看美國本土範圍，並從中抽樣 25 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 25 個（如果不足 25，就全部使用）
n_sample = min(25, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 25 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


## AutoARIMA

In [14]:
# ===== Cell 1: AutoARIMA 選出每個格點的 (p,d,q) 與 (P,D,Q,s) =====
from darts import TimeSeries
from darts.models.forecasting.sf_auto_arima import AutoARIMA
import pandas as pd

print("\n===== AutoARIMA order selection on sampled US cells (monthly data) =====")

# 1. 建 target DataFrame：n_sample 個格點 × T 時間點（原始尺度）
SAMPLE_SIZE = n_sample          # 預期是 25
T = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,                 # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)  # (T, SAMPLE_SIZE)

# 2. 切 train（AutoARIMA 用 train 段來選 order）
train_frac = 0.7
cut_train = int(T * train_frac)
split_1_time = ts_df.index[cut_train]

ts_all = TimeSeries.from_dataframe(ts_df)
train_ts, _ = ts_all.split_before(split_1_time)

print("Train length (raw for AutoARIMA):", len(train_ts))


# --- helper：依照 Nixtla 官方建議，從 statsforecast 的 model_ 把 order 解出來 ----
def get_orders_from_statsforecast_model(model_dict):
    """
    model_dict: sf_autoarima.model_，裡面有 "arma" 這個 key。
    依照 Nixtla 給的寫法，把 (p,d,q,P,D,Q,m) re-index 出來，並回傳：
      order          = (p, d, q)
      seasonal_order = (P, D, Q, m)   # m = season_length
    """
    # 參考 StackOverflow / Nixtla 寫法
    order = tuple(model_dict["arma"][i] for i in [0, 5, 1, 2, 6, 3, 4])
    m = int(order[6])
    order1 = (int(order[0]), int(order[1]), int(order[2]))
    seasonal_order = (int(order[3]), int(order[4]), int(order[5]), m)
    return order1, seasonal_order


# 3. 逐格點跑 darts.AutoARIMA，抓出 non-seasonal + seasonal order
orders = []
SEASON_LENGTH = 12   # 月資料 → 一年一季節

for i, col in enumerate(ts_df.columns):
    print(f"[{i+1}/{SAMPLE_SIZE}] Fitting AutoARIMA for {col} ...")

    # 單一格點的訓練序列 (TimeSeries)
    series_train_i = train_ts.univariate_component(col)

    # 建立 darts 的 AutoARIMA
    model_i = AutoARIMA(
        season_length=SEASON_LENGTH,  # 月資料：一年 12 期
        max_p=5,
        max_d=2,
        max_q=5,
        max_P=2,
        max_D=1,
        max_Q=2,
        # 如需可再加 information_criterion="aic" 等
    )

    # 在 train 段上 fit
    model_i.fit(series_train_i)

    # 取得底層 statsforecast AutoARIMA 物件
    sf_autoarima = model_i.model  # statsforecast.models.AutoARIMA 實例

    # 用 helper 拿到 (p,d,q) 和 (P,D,Q,s)
    (p, d, q), (P, D, Q, s) = get_orders_from_statsforecast_model(sf_autoarima.model_)

    orders.append({
        "cell": col,
        "p": p,
        "d": d,
        "q": q,
        "P": P,
        "D": D,
        "Q": Q,
        "s": s,
    })

orders_df = pd.DataFrame(orders)

print("\n===== AutoARIMA selected orders for each cell =====")
print(orders_df.to_string(index=False))



===== AutoARIMA order selection on sampled US cells (monthly data) =====
Total time steps: 548
ts_df shape: (548, 25)
Train length (raw for AutoARIMA): 383
[1/25] Fitting AutoARIMA for cell_0 ...
[2/25] Fitting AutoARIMA for cell_1 ...
[3/25] Fitting AutoARIMA for cell_2 ...
[4/25] Fitting AutoARIMA for cell_3 ...
[5/25] Fitting AutoARIMA for cell_4 ...
[6/25] Fitting AutoARIMA for cell_5 ...
[7/25] Fitting AutoARIMA for cell_6 ...
[8/25] Fitting AutoARIMA for cell_7 ...
[9/25] Fitting AutoARIMA for cell_8 ...
[10/25] Fitting AutoARIMA for cell_9 ...
[11/25] Fitting AutoARIMA for cell_10 ...
[12/25] Fitting AutoARIMA for cell_11 ...
[13/25] Fitting AutoARIMA for cell_12 ...
[14/25] Fitting AutoARIMA for cell_13 ...
[15/25] Fitting AutoARIMA for cell_14 ...
[16/25] Fitting AutoARIMA for cell_15 ...
[17/25] Fitting AutoARIMA for cell_16 ...
[18/25] Fitting AutoARIMA for cell_17 ...
[19/25] Fitting AutoARIMA for cell_18 ...
[20/25] Fitting AutoARIMA for cell_19 ...
[21/25] Fitting AutoAR

## 用 Auto ARIMA 做 SARIMA

In [15]:
# ===== Cell 2: 用 AutoARIMA 選出的 order 對每格點跑 SARIMA，並算 VAL/TEST 指標 =====
import numpy as np
from darts import TimeSeries
from darts.models import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== SARIMA (per-cell AutoARIMA full orders) on sampled US cells (monthly) =====")

SAMPLE_SIZE = ts_df.shape[1]
T = ts_df.shape[0]

# 1. 建 70 / 15 / 15 切分（train / val / test）
train_frac = 0.7
val_frac   = 0.15

cut_train = int(T * train_frac)
cut_val   = int(T * (train_frac + val_frac))

idx = ts_df.index
split_1_time = idx[cut_train]
split_2_time = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 2. 用 train 統計量標準化（只標 target，自然無 covariate）
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled  = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)
val_true_raw  = vals_all[cut_train:cut_val, :]
test_true_raw = vals_all[cut_val:, :]

def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values

print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 3. 逐格點跑 SARIMA（使用各自 AutoARIMA 選出的 (p,d,q,P,D,Q,s)）
pred_sar_val_scaled  = np.zeros((T_val,  SAMPLE_SIZE), dtype=np.float32)
pred_sar_test_scaled = np.zeros((T_test, SAMPLE_SIZE), dtype=np.float32)

for i, col in enumerate(ts_df.columns):
    # 讀出該 cell 的 order
    row = orders_df[orders_df["cell"] == col].iloc[0]
    p_i, d_i, q_i = int(row["p"]), int(row["d"]), int(row["q"])
    P_i, D_i, Q_i = int(row["P"]), int(row["D"]), int(row["Q"])
    s_i           = int(row["s"])

    print(f"[{i+1}/{SAMPLE_SIZE}] Fitting SARIMA for {col} with "
          f"(p,d,q)=({p_i},{d_i},{q_i}), (P,D,Q,s)=({P_i},{D_i},{Q_i},{s_i}) ...")

    # (A) VAL：train-only → 預測 val
    series_train_i = train_ts.univariate_component(col)

    model_sarima_val_i = ARIMA(
        p=p_i,
        d=d_i,
        q=q_i,
        seasonal_order=(P_i, D_i, Q_i, s_i),
    )
    model_sarima_val_i.fit(series_train_i)

    pred_val_i = model_sarima_val_i.predict(n=T_val)

    pred_val_i_vals = np.asarray(pred_val_i.all_values(copy=False))
    if pred_val_i_vals.ndim == 3:
        pred_val_i_vals = pred_val_i_vals[..., 0]
    pred_val_i_vals = pred_val_i_vals.reshape(-1)

    pred_sar_val_scaled[:, i] = pred_val_i_vals

    # (B) TEST：train+val → 預測 test
    series_val_i       = val_ts.univariate_component(col)
    series_train_val_i = series_train_i.concatenate(series_val_i)

    model_sarima_test_i = ARIMA(
        p=p_i,
        d=d_i,
        q=q_i,
        seasonal_order=(P_i, D_i, Q_i, s_i),
    )
    model_sarima_test_i.fit(series_train_val_i)

    pred_test_i = model_sarima_test_i.predict(n=T_test)

    pred_test_i_vals = np.asarray(pred_test_i.all_values(copy=False))
    if pred_test_i_vals.ndim == 3:
        pred_test_i_vals = pred_test_i_vals[..., 0]
    pred_test_i_vals = pred_test_i_vals.reshape(-1)

    pred_sar_test_scaled[:, i] = pred_test_i_vals

print("SARIMA (AutoARIMA orders) val pred (scaled)  shape:", pred_sar_val_scaled.shape)
print("SARIMA (AutoARIMA orders) test pred (scaled) shape:", pred_sar_test_scaled.shape)

# 4. 反標準化
pred_sar_val_raw  = inverse_scale(pred_sar_val_scaled)
pred_sar_test_raw = inverse_scale(pred_sar_test_scaled)

print("pred_sar_val_raw shape:", pred_sar_val_raw.shape)
print("pred_sar_test_raw shape:", pred_sar_test_raw.shape)

# 5. 指標
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_sar_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_sar_test_raw.reshape(-1, SAMPLE_SIZE)

mse_sar_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_sar_val  = mean_absolute_error(y_val_true, y_val_pred)
mse_sar_test = mean_squared_error(y_test_true, y_test_pred)
mae_sar_test = mean_absolute_error(y_test_true, y_test_pred)

rmse_sar_val  = np.sqrt(mse_sar_val)
rmse_sar_test = np.sqrt(mse_sar_test)

r2_sar_val  = r2_score(y_val_true,  y_val_pred)
r2_sar_test = r2_score(y_test_true, y_test_pred)

print("\n===== SARIMA RESULTS with AutoARIMA (monthly, sampled cells) =====")
print(f"VAL  RMSE: {rmse_sar_val:.4f}, MSE: {mse_sar_val:.4f}, MAE: {mae_sar_val:.4f}, R²: {r2_sar_val:.4f}")
print(f"TEST RMSE: {rmse_sar_test:.4f}, MSE: {mse_sar_test:.4f}, MAE: {mae_sar_test:.4f}, R²: {r2_sar_test:.4f}")

metrics_table = pd.DataFrame({
    "Model": ["SARIMA_AutoARIMA_full", "SARIMA_AutoARIMA_full"],
    "Split": ["VAL",                   "TEST"],
    "RMSE":  [rmse_sar_val,            rmse_sar_test],
    "MSE":   [mse_sar_val,             mse_sar_test],
    "MAE":   [mae_sar_val,             mae_sar_test],
    "R2":    [r2_sar_val,              r2_sar_test],
})

print("\nMSE / MAE / RMSE / R² table (SARIMA with per-cell full AutoARIMA orders, monthly):")
print(metrics_table.to_string(index=False))



===== SARIMA (per-cell AutoARIMA full orders) on sampled US cells (monthly) =====
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled TimeSeries lengths]
T_train = 383 T_val = 82 T_test = 83
[1/25] Fitting SARIMA for cell_0 with (p,d,q)=(1,0,0), (P,D,Q,s)=(2,1,1,12) ...
[2/25] Fitting SARIMA for cell_1 with (p,d,q)=(0,0,2), (P,D,Q,s)=(1,1,1,12) ...
[3/25] Fitting SARIMA for cell_2 with (p,d,q)=(0,0,2), (P,D,Q,s)=(0,1,1,12) ...
[4/25] Fitting SARIMA for cell_3 with (p,d,q)=(1,0,1), (P,D,Q,s)=(0,1,1,12) ...
[5/25] Fitting SARIMA for cell_4 with (p,d,q)=(1,0,1), (P,D,Q,s)=(1,1,1,12) ...
[6/25] Fitting SARIMA for cell_5 with (p,d,q)=(1,0,0), (P,D,Q,s)=(2,1,1,12) ...
[7/25] Fitting SARIMA for cell_6 with (p,d,q)=(1,0,0), (P,D,Q,s)=(1,1,1,12) ...
[8/25] Fitting SARIMA for cell_7 with (p,d,q)=(1,0,1), (P,D,Q,s)=(2,1,1,12) ...
[9/25] Fitting SARIMA for cell_8 with (p,d,q)=(1,0,1), (P,D,Q,s)=(0,1,1,12) ...
[10/25] Fitting SARIMA for cell_9 with (p,d,q)=(1,0,0), (P,D,Q,s)=(1,1,1,12) ...
[1

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[12/25] Fitting SARIMA for cell_11 with (p,d,q)=(1,0,0), (P,D,Q,s)=(1,1,1,12) ...
[13/25] Fitting SARIMA for cell_12 with (p,d,q)=(2,0,2), (P,D,Q,s)=(2,1,1,12) ...


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimizat

[14/25] Fitting SARIMA for cell_13 with (p,d,q)=(0,0,3), (P,D,Q,s)=(0,1,1,12) ...
[15/25] Fitting SARIMA for cell_14 with (p,d,q)=(0,0,2), (P,D,Q,s)=(0,1,1,12) ...
[16/25] Fitting SARIMA for cell_15 with (p,d,q)=(2,0,0), (P,D,Q,s)=(1,1,1,12) ...
[17/25] Fitting SARIMA for cell_16 with (p,d,q)=(0,0,2), (P,D,Q,s)=(0,1,1,12) ...
[18/25] Fitting SARIMA for cell_17 with (p,d,q)=(2,0,1), (P,D,Q,s)=(1,1,1,12) ...


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[19/25] Fitting SARIMA for cell_18 with (p,d,q)=(1,0,0), (P,D,Q,s)=(0,1,2,12) ...
[20/25] Fitting SARIMA for cell_19 with (p,d,q)=(1,0,1), (P,D,Q,s)=(1,1,1,12) ...
[21/25] Fitting SARIMA for cell_20 with (p,d,q)=(2,0,0), (P,D,Q,s)=(2,1,1,12) ...
[22/25] Fitting SARIMA for cell_21 with (p,d,q)=(1,0,0), (P,D,Q,s)=(1,1,1,12) ...
[23/25] Fitting SARIMA for cell_22 with (p,d,q)=(0,0,1), (P,D,Q,s)=(1,1,1,12) ...
[24/25] Fitting SARIMA for cell_23 with (p,d,q)=(3,0,0), (P,D,Q,s)=(2,1,1,12) ...
[25/25] Fitting SARIMA for cell_24 with (p,d,q)=(0,0,2), (P,D,Q,s)=(0,1,1,12) ...
SARIMA (AutoARIMA orders) val pred (scaled)  shape: (82, 25)
SARIMA (AutoARIMA orders) test pred (scaled) shape: (83, 25)
pred_sar_val_raw shape: (82, 25)
pred_sar_test_raw shape: (83, 25)

===== SARIMA RESULTS with AutoARIMA (monthly, sampled cells) =====
VAL  RMSE: 1.7723, MSE: 3.1409, MAE: 1.3049, R²: 0.9201
TEST RMSE: 1.6430, MSE: 2.6994, MAE: 1.2301, R²: 0.9324

MSE / MAE / RMSE / R² table (SARIMA with per-cell full A

### SARIMA (AutoARIMA Full Orders, Monthly) — 25 Sampled US Grid Cells

**Data Summary**  
- Frequency: **Monthly SARIMA** (s = 12)  
- Total time steps: **548**  
- Train length: **383**  
- Validation length: **82**  
- Test length: **83**  
- Grid cells: **25**  
- Each cell fitted with **its own AutoARIMA-selected (p,d,q)(P,D,Q,12)**
---

### Performance Summary

| Model                        | Split |   RMSE   |   MSE    |   MAE   |    R²    |
|:-----------------------------|:-----:|:--------:|:--------:|:-------:|:--------:|
| SARIMA_AutoARIMA_full        |  VAL  | 1.772251 | 3.140874 | 1.304855 | 0.920104 |
| SARIMA_AutoARIMA_full        | TEST  | 1.642992 | 2.699424 | 1.230132 | 0.932416 |

